## General questions to be addressed by this project
How to take down bitcoin
In what ways do the incentives we see in bitcoin get us to attack or fault tolerance that mirrors what we see in other replicated log problems.

We say that chandra toueg failure detectors can handle single failure running CT consensus. 
Then Paxos can handle f of 2f+1. But, Bitcoin network cannot handle anywhere near that amount. 

So check on the known failures 

Then, check on the way that collisions come up at block generation time for miners that have otherwise the same keys? what happens to their block and can they append it to the new chain or would credit be robbed from them if someone else appends it. Credit must be at append stage otherwise no incentive to append (other than protect ur blocks before from being overwritten) but that makes it difficult to think about credit being undone when in the real chain something else is produced. 
Maybe you can never be verified if you don't stick with the real chain. Hence, you always have incentive to add to the real chain rather than creating your own little fake blocks at the end of a fork you've created. 

# Papers read
Read Satoshi's original BTC paper:
Learned
- While a replicated, consistent log for google or others that we have looked at allows for application functionality, a replicated consistent log for btc allows for a secure value store.
  - The value to btc is based on compute cycles, the security is also based on compute cycles (you would need to spend an ungodly amount of compute cycles very quickly to move faster than the rest of the network. As long as you are not moving faster than the rest of the network, your actions are meaningless.)
  - Cryptography seperates valid from invalid (nodes, blocks) but PoW separates canonical from discarded. Nothing invalid is valuable. You need both validity and value to interfere with the network
- The affect of this incentive-based security is that a decentralized ledger can be treated as cannonical (and almost central from transactions that are old enough that they cannot be undone) 
- For miners, rewards come both from subsidies and fees. The subsidies are on a decreasing schedule, but after each subsidy decrease, market price has gone up so each btc is more expensive. 
- The idea is that it is in each miners economic interest to mine and show the block when they find it and add it to the longest chain (follow the btc protocol). 
- When this breaks is when rewards do not correspond evenly to compute which is a premise of btc.
Read Israel Crypto Expert (Selfish-mine)
- Argued that the 51% threshold is not the true safe threshold. If nodes can take advantage of being on the same length of chain as the main pool, by not revealing if go two ahead until are one ahead, revealing if one ahead or 0 ahead and getting the main chain to build on theirs, pools of miners can gain an outsized reward.
- This is in large part determined by potential to get ambivalent nodes to mine on their tip rather than the tip of the main chain. This seems quite hard, but in practice if you can propagate messages across the network faster than the node who just built a block can, can find success.
- This paper found that the threshold for a pool attack being strongly successful was around 25% of nodes involved.
- Suggested a fix where on incoming chains, flip fair coin to decide between the two. This way, you never have more than 50% of nodes in expectation mining on the "bad branch" (but you do take some loss if the bad branch is the second to arrive)
GitHub Codebase 
- The protocol has a github and to be ammended through soft forks, which users can change by themselves, or by hard forks which must be done throughout the community if there is a big change. If a hard fork is not agreeed upon, then a partition of the network forms
- Bitcoin core is the protocol and is managed by a handful of people, not updated very often

BFG attacks reading
Eclipse attacks reading
Looked at some btc mining tracking websites
- Not truly evenly distributed as I thought
  - China controls 41% of the compute
  - Foundry (us company) controls 31%
- There are 1,000,000 btc left
- About every 10 min a block is mined but there is significant variance there depending on fullness of the mempoool from which transactions are pulled
- 2150 is year we will stop having fees
  

Was going to go totally no sim, but then the question I was trying to answer, of "how many pool block forwarding nodes do you need to disrupt network dynamics" seemed tough.

I thought anything within distance 2 on the graph of the network might get the honest result, greater than that gets the fake result. Given that btc nodes can have approx 120 incoming connections, 8 outgoing connections (not sure where I pulled this from) doesn't seem like you need too many shadow (watcher) nodes. If a graph is even 1000 nodes big, 2 waves covers less than 10% and gives the attacking pool 90% $\gamma$ which is more than enough than needed by the papers arg.

But, I decided that the report will look like:
1) Known attacks
2) Economic Breakdowns (network changes and attacks)
3) Economic Breakdowns (fee and subsidy changes)

The failures truly do come down to when are nodes incentivized to behave against the protocol. Because, the protocol is simple -> safe and most of it's power comes from the spread of the flops behind it (no actor controls enough to influence the other). But, when actors are incentivized not to follow protocol this is a first level failure, when significant deviation occurs from the distribution of value that should occur given compute power per node, this is a second level failure.

I will lay different failure models out (none of which break the network) that break btc. Then, sim will evaluate how bad of a failure it is and we can do some math for why + extrapolation.
NOTE: How bad is the abs diff btwn actual reward and theoretical reward if all nodes were honest and running bitcoin core.

## Sim (Claude)

> Reference description of the simulator currently under construction. What it does, why it does it that way, what's coming next. Use this as the orientation doc when picking up on Wednesday.

### Framing

The simulator models Bitcoin-style consensus at the **strategic-incentive layer** rather than the message/channel layer. The fault model is not "messages get dropped"; it is "rational actors deviate from the protocol when it pays them to." The headline question of the project is:

> Under what protocol parameters and network conditions does honest behavior remain the dominant strategy, and how badly does the realized reward distribution diverge from the fair-share distribution when it doesn't?

This is mechanism design. The protocol designer chooses rules (block reward schedule, fee mechanism, block capacity, tie-break rule); strategic players respond by choosing strategies; the equilibrium tells us whether the rules hold up. The simulator is the tool for measuring it.

### Computational model

- **Discrete-event simulation (DES) with virtual time.** A priority queue of `(timestamp, action)` pairs drives the simulator. Virtual time advances in jumps to the next scheduled event &mdash; no real-time waiting, no fixed time-steps. Runs are deterministic given a seed and proceed as fast as the CPU consumes events.
- **Mining as a Poisson process.** Each miner $i$ with compute share $c_i$ produces blocks via an exponential clock of rate $c_i$. We sample the next block-finding time once when scheduling, and resample on tip changes. This is statistically identical to the continuous-time picture because the exponential is memoryless &mdash; partial mining progress does not need to be tracked.
- **Two-channel peering.** Every node has a `public_peers` list (the normal gossip graph) and a `coalition_peers` list (out-of-band coordination between colluders). Honest nodes have empty coalition channels. The dual-channel design is what makes selfish mining and poison-helper coordination representable without special cases.
- **Per-node chain views.** Every node holds its own `unordered_map<hash, Block>` plus a tip pointer. The "canonical chain" is whatever the most-work tip is from the global observer's perspective. Reorgs are local: when a heavier chain arrives, a node's tip moves; nothing global needs to coordinate.
- **Lazy reward accounting.** No per-event reward bookkeeping. At measurement time, walk the canonical chain and tally coinbases per miner. Reorged-out blocks naturally contribute zero because they are not on the walked path. This makes reorg correctness automatic.

### The leaderboard metric

For node $i$ over horizon $T$:

$$\mathrm{score}_i(T) \;=\; \frac{\text{reward earned by } i \text{ during } [0,T]}{\dfrac{c_i}{\sum_j c_j}\cdot R\cdot T}$$

where $R$ is total reward emitted per unit time. The denominator is node $i$'s fair share &mdash; what they would earn in a fully honest world. Score $= 1$ means earning fair share; $> 1$ means strategic gain; $< 1$ means strategic loss. By the law of large numbers, scores converge to their equilibrium values with standard error $\sim 1/\sqrt{T}$.

This is exactly the *relative revenue* metric from Eyal & Sirer (2014). It gives us a built-in validation: running a selfish-mining strategy in this simulator should reproduce their classical threshold curve at $\alpha \approx 0.25$. If it doesn't, the simulator's foundations are wrong.

### Node taxonomy

Three populations, but the same underlying `Node` struct for all of them. Behavior is dispatched on two enums (`MiningStrategy`, `PoisonStrategy`), not via class inheritance.

| Population         | Compute | `MiningStrategy`  | `PoisonStrategy` | `coalition_peers` |
|--------------------|:-------:|-------------------|------------------|-------------------|
| Honest miner       |  $> 0$  | `Honest`          | `None`           | empty             |
| Colluding miner    |  $> 0$  | `SelfishMining`*  | `None`           | other colluders   |
| Poison "signpost"  |   $0$   | `None`            | `Eclipse`*       | colluders served  |

\* Reserved enum values, not yet implemented.

All three share infrastructure (chain view, event handlers, message processing). Adding a new attack is an enum value plus a few case branches &mdash; not a new class hierarchy.

The poison nodes are not paid by the protocol; their justification is that the malicious coalition pays for their placement out-of-band. The economic unit of interest is therefore the coalition's *aggregate* score across its miners.

### Paper sections as experimental sweeps

The three planned report sections are three sweeps over the same simulator. Measurements are identical across them; only the parameter being varied changes.

| Section | Held fixed              | Swept                                                       | Measured                                |
|---------|-------------------------|-------------------------------------------------------------|-----------------------------------------|
| 1. Known attacks      | topology, fee mechanism | attacker hashpower $\alpha$, attack type             | leaderboard, reorg rate                 |
| 2. Network structure  | attack, fee mechanism   | poison node count and placement, topology density    | leaderboard, reorg rate, eclipse efficacy |
| 3. Fee structure      | attack, topology        | subsidy schedule, fee rule, block capacity           | leaderboard, fee revenue distribution   |

The simulator surface this implies: fill in a parameter table, run $N$ replicates, plot the result column. The pieces of code that vary across sections are small; the bulk of the codebase is shared.

### Code layout (v0)

```
src/
├── types.hh         NodeId, BlockHash, Time, EventId, GENESIS_HASH
├── chain.hh / .cc   Block, NodeChainView (per-node chain view with first-seen tiebreak)
├── node.hh          Node struct, MiningStrategy + PoisonStrategy enums, dual-channel peers
├── simulator.hh /.cc DES core: priority queue, lazy cancellation, gossip, leaderboard
└── run_sim.cc       v0 driver: N honest miners, fully-connected, prints leaderboard
```

### What v0 does today

- DES core with lazy cancellation.
- Block + chain data structures with first-seen tiebreak.
- Honest mining: schedule `Exp(c_i)` block-found event; on fire, build block from current tip and gossip; on receive, validate, add, restart mining if tip changed.
- Gossip cascade with deterministic latency, terminating when peers already know a block.
- End-of-run leaderboard report (walks the canonical chain).

### What v0 does *not* do (intentionally)

- No selfish mining; the `SelfishMining` enum value exists but is unhandled.
- No poison nodes; `Eclipse` exists but is unhandled.
- No mempool, transactions, fees, or block capacity.
- No topology variation (fully-connected peer graph).
- No replicate-driven statistics or config-file loading; parameters are constants in `run_sim.cc`.

### v0 acceptance test

With all-honest miners and gossip latency $\ll$ mean block interval, every score should converge to $1.0$ within $1/\sqrt{T}$. If a long run (say $T = 10{,}000$) produces scores in $[0.97, 1.03]$ for every miner, the foundations are correct and we can start layering attacks on top. If not, something in the DES dispatch, the gossip cascade, or the tip-update logic is wrong.

### Roadmap

- **v1.** Selfish mining + eclipse attacks. Reproduces classical literature results. Sufficient for paper section 1 and (with topology variation) most of section 2.
- **v2.** Mempool + transaction generator + flat fees + bounded block capacity. Mempool clearing on apply, restoration on reorg. Sufficient for an early section 3.
- **v3.** Variable-fee market, alternative reward schedules (geometric decay, fee-only à la Carlsten et al., EIP-1559-style burn). Sufficient for full section 3, including the "fee-only Bitcoin is unstable" thesis.

### Design decisions worth re-reading on Wednesday

1. **Virtual time, not real time.** The DES priority queue is the clock. No `sleep`, no wall-clock dependence; runs are deterministic given a seed.
2. **Memorylessness handles tip changes.** When a node's tip moves, cancel the outstanding mining event and resample. Partial progress does not matter. This only works because the exponential is memoryless &mdash; a non-exponential mining-time distribution would require explicit progress tracking.
3. **Block contents are decided at fire time, not schedule time.** Matters once there is a mempool: the miner uses whatever transactions are in their mempool at the moment the block is found, not when mining started. Decoupling mining time from content selection lets us add the mempool layer later without touching the DES.
4. **Rewards aren't tracked during simulation.** They are derived from the canonical chain at the end. Reorged blocks contribute zero by construction.
5. **Two peering channels, not one.** Strategic coordination among colluders happens out-of-band; the second channel is the model of that, with no propagation to public peers. This is what makes selfish mining cleanly expressible.
6. **Strategy is an enum, not a class hierarchy.** Adding a new attack is one enum value plus a small number of switch cases (assemble, on found, on received). The infrastructure does not move.

### Open questions to revisit Wednesday

- **Canonical chain in the global view.** Currently using node 0's tip as a proxy, which is fine under honest play with low latency. Under attack, we want an explicit `Simulator::canonical_chain()` that returns the most-work tip across all node views.
- **Poison node placement policy.** Random / targeted at high-compute honest miners / targeted at high-degree honest miners / targeted at one designated victim. Each gives different leverage; section 2 wants to sweep over them.
- **Confirmation depth $k$ for double-spend success.** Standard is 6; for short simulations 6 might be too deep to resolve. Parameterize.
- **Network topology generator.** Random regular vs. preferential attachment vs. small-world. Section 2 wants to compare; need a topology factory.
- **Replicate harness.** v0 runs one simulation per invocation. For real experiments we want $n$ replicates with confidence intervals, ideally with seeds chosen so that runs are reproducible.
